# Grokking Reproduction — Nanda et al. (2023)
**Paper:** Progress Measures for Grokking via Mechanistic Interpretability  
**DOI:** 10.48550/arXiv.2301.05217  

Run cells in order. Each cell is independent and re-runnable after a `git pull`.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/MrFaruk0/grokking-reproduction.git"
REPO_DIR = Path("/content/grokking-reproduction")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.insert(0, str(REPO_DIR))
!pip install einops plotly pandas -q
Path("outputs").mkdir(exist_ok=True)
print("Setup complete.")

In [ ]:
print(Path("PROJECT_MEMORY.md").read_text())

In [ ]:
from transformers import Config, train_model

config = Config(
    p=113, d_model=128, num_heads=4, d_mlp=512,
    num_layers=1, frac_train=0.3, num_epochs=10000,
    seed=0, act_type='ReLU', use_ln=False, fn_name='add',
)
world = train_model(config)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs = list(range(0, len(world.train_accs) * 100, 100))
grok_ep = next((e for e, a in zip(epochs, world.test_accs) if a >= 0.95), None)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs, world.train_accs, color='#4C72B0', lw=2, label='Train accuracy')
ax.plot(epochs, world.test_accs,  color='#DD8452', lw=2, label='Test accuracy')
if grok_ep:
    ax.axvline(grok_ep, color='#C44E52', linestyle='--', alpha=0.6,
               linewidth=1.5, label=f'Grokking epoch ({grok_ep})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_ylim(-0.05, 1.05)
ax.set_title(
    'Figure 1: Grokking Curve — Train vs. Test Accuracy\n'
    'Modular addition (a+b) mod 113 · 1-layer Transformer · AdamW (lr=1e-3, wd=1.0)',
    fontsize=14)
ax.legend(fontsize=10)
fig.tight_layout()
fig.savefig('outputs/figure1_grokking_curve.png', dpi=150, bbox_inches='tight')
fig.savefig('outputs/figure1_grokking_curve.pdf', bbox_inches='tight')
plt.show()
print(f'Grokking epoch: {grok_ep}')
print(f'Final train acc: {world.train_accs[-1]:.4f}')
print(f'Final test acc:  {world.test_accs[-1]:.4f}')

In [ ]:
from figures import plot_fourier_spectrum
plot_fourier_spectrum(world.model, config, save_dir='outputs')

In [ ]:
from explorations import sweep_weight_decay
from figures import plot_weight_decay_sweep

wd_results = sweep_weight_decay(config, weight_decays=[0.0, 0.1, 0.5, 1.0, 2.0])
plot_weight_decay_sweep(wd_results, save_dir='outputs')

In [ ]:
from explorations import sweep_prime_p
from figures import plot_p_sweep

p_results = sweep_prime_p(config, primes=[53, 97, 113, 127])
plot_p_sweep(p_results, save_dir='outputs')

In [ ]:
from explorations import sweep_operations
from figures import plot_operations_sweep

op_results = sweep_operations(config, operations=['add', 'subtract', 'multiply'])
plot_operations_sweep(op_results, save_dir='outputs')

In [ ]:
from explorations import sweep_depth
from figures import plot_depth_sweep

depth_results = sweep_depth(config, depths=[1, 2, 3])
plot_depth_sweep(depth_results, save_dir='outputs')